In [ ]:
# 필요한 패키지 불러오기
import pandas as pd
import time
import os
import random
from selenium.webdriver.common.by import By
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from openpyxl import Workbook
from bs4 import BeautifulSoup
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import NoSuchElementException


In [ ]:
# # Webdriver headless mode setting
# def get_chrome_options():
#     options = webdriver.ChromeOptions()
#     # options.add_argument("--headless=new") # headless 모드: 실제 창이 뜨지 않은 채 메모리상에서만 브라우저 구동
#     options.add_argument('window-size=1920x1080') # 브라우저의 가상 창 크기 설정: 창 크기 명시하지 않으며 요소(버튼 등)가 겹쳐서 클릭이 안되는 문제 있을 수 있음 
#     options.add_argument("disable-gpu") # 그래픽 카드(GPU) 가속 기능 끔. 과거 헤드리스 모드에서 GPU 가속 활성화돼 있으면 오류 발생하는 버그 있었음
#     return options

# # BS4 setting for secondary access
# session = requests.Session()
# headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"}

# retries = Retry(total=5,
#                 backoff_factor=0.1, # 재시도 사이의 대기 시간
#                 status_forcelist=[500, 502, 503, 504])

# session.mount('https://', HTTPAdapter(max_retries=retries))
# # session.mount('http://', HTTPAdapter(max_retries=retries))

In [ ]:
# SELECT_KEYWORDS = [
#     "분위기가 좋아요", "인테리어", "조용한", "트렌디", "데이트", "특별한 날", "가족", "가성비"
# ]
# COMMENT_KEYWORDS = [
#     "혼밥", "국물", "따뜻한", "파전", "아늑한",
#     "비오는 날", "이자카야", "전골", "시원한", "여름", "냉면", "빙수",
#     "브런치", "테라스", "창가", "날씨 좋은", "신선한", "안주"
# ]
# OUTPUT_FILE = "naver_keyword_count.csv"

# def get_keyword_counts(restaurant_name, rest_id, picture, options):
#     service = Service(ChromeDriverManager().install())
#     driver = webdriver.Chrome(service=service, options=options)
#     driver.get(f"https://m.place.naver.com/restaurant/{rest_id}/review/visitor")
#     driver.implicitly_wait(30)
 
#     # --- 식당 카테고리 ---
#     category = "아무거나"
#     try:
#         category_el = driver.find_element(By.CLASS_NAME, 'lnJFt')
#         category = category_el.text.strip()
#         print(f"  [카테고리] {category}")
#     except NoSuchElementException:
#         print(f"  [카테고리] 요소를 찾지 못했습니다.")
#     except Exception as e:
#         print(f"  [카테고리] 수집 에러: {e}")
 
#     driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.END)
#     time.sleep(1.0)
 
#     target2_clicked = False
#     try:
#         while True:
#             driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)
#             try:
#                 target = driver.find_element(By.CLASS_NAME, 'TeItc')
#                 driver.execute_script("arguments[0].click();", target)
#             except NoSuchElementException:
#                 if not target2_clicked:
#                     print(f"  [페이지] target 없음 → target2 클릭")
#                     target2 = driver.find_element(By.CLASS_NAME, 'Kv9y6')
#                     driver.execute_script("arguments[0].click();", target2)
#                     target2_clicked = True
#                 else:
#                     break
#     except Exception as e:
#         print(f"  [페이지] 스크롤 에러: {e}")
 
#     # ── 키워드 카운팅 초기화 ───────────────────────
#     row = {
#         "식당명": restaurant_name,
#         "카테고리": category,
#         "picture": picture,  # CSV에서 가져온 이미지 URL 저장
#     }
#     for keyword in SELECT_KEYWORDS + COMMENT_KEYWORDS:
#         row[keyword] = 0
 
#     # ── 더보기 버튼 클릭 ──────────────────────────
#     try:
#         more_btns = driver.find_elements(By.CSS_SELECTOR, ".pui__vn15t2")
#         print(f"  [더보기] {len(more_btns)}개 버튼 클릭")
#         for btn in more_btns:
#             try:
#                 driver.execute_script("arguments[0].click();", btn)
#                 time.sleep(0.3)
#             except Exception as e:
#                 print(f"  [더보기] 버튼 클릭 에러: {e}")
#     except Exception as e:
#         print(f"  [더보기] 에러: {e}")
 
#     # ── SELECT_KEYWORDS 파싱 ───────────────────────
#     try:
#         keyword_els = driver.find_elements(By.CSS_SELECTOR, ".pui__jhpEyP")
#         print(f"  [SELECT] 키워드 요소 {len(keyword_els)}개 수집")
#         for el in keyword_els:
#             keyword_text = el.text.strip()
#             for keyword in SELECT_KEYWORDS:
#                 if keyword in keyword_text:
#                     row[keyword] += 1
#     except Exception as e:
#         print(f"  [SELECT] 파싱 에러: {e}")
 
#     # ── COMMENT_KEYWORDS 파싱 ─────────────────────
#     try:
#         comments = driver.find_elements(By.CSS_SELECTOR, ".pui__vn15t2")
#         print(f"  [댓글] {len(comments)}개 수집")
#         for el in comments:
#             comment_text = el.text
#             for keyword in COMMENT_KEYWORDS:
#                 if keyword in comment_text:
#                     row[keyword] += 1
#     except Exception as e:
#         print(f"  [댓글] 파싱 에러: {e}")
 
#     finally:
#         driver.quit()
 
#     return row
 
 
# def load_existing_results(output_file):
#     """이미 저장된 결과 불러오기 → 완료된 식당명 집합 반환"""
#     if os.path.exists(output_file):
#         try:
#             df_existing = pd.read_csv(output_file, encoding="utf-8-sig")
#             done = set(df_existing["식당명"].tolist())
#             print(f"✅ 기존 결과 파일 로드: {len(done)}개 식당 이미 완료\n")
#             return df_existing, done
#         except Exception as e:
#             print(f"⚠️ 기존 파일 읽기 실패 ({e}) → 새로 시작\n")
#     return pd.DataFrame(), set()

In [ ]:
# ── 키워드 정의 ───────────────────────────────────
SELECT_KEYWORDS = [
    "분위기가 좋아요", "인테리어", "조용한", "트렌디", "데이트", "특별한 날", "가족", "가성비"
]
COMMENT_KEYWORDS = [
    "혼밥", "국물", "따뜻한", "파전", "아늑한",
    "비오는 날", "이자카야", "전골", "시원한", "여름", "냉면", "빙수",
    "브런치", "테라스", "창가", "날씨 좋은", "신선한", "안주"
]

OUTPUT_FILE = "naver_keyword_count.csv"

# ── ChromeDriver 한 번만 설치 ────────────────────
service = Service(ChromeDriverManager().install())


def get_chrome_options():
    options = webdriver.ChromeOptions()
    # options.add_argument("--headless=new")       # 창 없이 실행하려면 주석 해제
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("window-size=1920x1080")

    # 모바일 UA — m.place.naver.com 셀렉터에 필수
    options.add_argument(
        "user-agent=Mozilla/5.0 (iPhone; CPU iPhone OS 16_0 like Mac OS X) "
        "AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Mobile/15E148 Safari/604.1"
    )

    # 봇 감지 우회
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    return options


def random_sleep(min_sec=1.5, max_sec=3.5):
    """랜덤 딜레이 — 봇 감지 방지"""
    t = random.uniform(min_sec, max_sec)
    time.sleep(t)


def get_keyword_counts(restaurant_name, rest_id, picture, options):
    driver = webdriver.Chrome(service=service, options=options)

    # navigator.webdriver 숨기기
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )

    driver.get(f"https://m.place.naver.com/restaurant/{rest_id}/review/visitor")
    driver.implicitly_wait(30)
    random_sleep(2.0, 4.0)  # 페이지 로드 후 대기

    # ── 카테고리 수집 ─────────────────────────────
    category = "아무거나"
    try:
        category_el = driver.find_element(By.CLASS_NAME, 'lnJFt')
        category = category_el.text.strip()
        print(f"  [카테고리] {category}")
    except NoSuchElementException:
        print(f"  [카테고리] 요소를 찾지 못했습니다.")
    except Exception as e:
        print(f"  [카테고리] 수집 에러: {e}")

    # ── 스크롤 & 더보기 클릭 ──────────────────────
    driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.END)
    random_sleep(1.0, 2.0)

    target2_clicked = False
    try:
        while True:
            driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.PAGE_DOWN)
            random_sleep(0.3, 0.8)  # 스크롤 간 딜레이
            try:
                target = driver.find_element(By.CLASS_NAME, 'TeItc')
                driver.execute_script("arguments[0].click();", target)
            except NoSuchElementException:
                if not target2_clicked:
                    print(f"  [페이지] target 없음 → target2 클릭")
                    target2 = driver.find_element(By.CLASS_NAME, 'Kv9y6')
                    driver.execute_script("arguments[0].click();", target2)
                    target2_clicked = True
                else:
                    break
    except Exception as e:
        print(f"  [페이지] 스크롤 에러: {e}")

    # ── 키워드 카운팅 초기화 ──────────────────────
    row = {
        "식당명": restaurant_name,
        "카테고리": category,
        "picture": picture,
    }
    for keyword in SELECT_KEYWORDS + COMMENT_KEYWORDS:
        row[keyword] = 0

    # ── 더보기 버튼 클릭 ──────────────────────────
    try:
        more_btns = driver.find_elements(By.CSS_SELECTOR, ".pui__vn15t2")
        print(f"  [더보기] {len(more_btns)}개 버튼 클릭")
        for btn in more_btns:
            try:
                driver.execute_script("arguments[0].click();", btn)
                random_sleep(0.5, 1.5)  # 클릭 간 랜덤 딜레이
            except Exception as e:
                print(f"  [더보기] 버튼 클릭 에러: {e}")
    except Exception as e:
        print(f"  [더보기] 에러: {e}")

    # ── SELECT_KEYWORDS 파싱 ──────────────────────
    try:
        keyword_els = driver.find_elements(By.CSS_SELECTOR, ".pui__jhpEyP")
        print(f"  [SELECT] 키워드 요소 {len(keyword_els)}개 수집")
        for el in keyword_els:
            keyword_text = el.text.strip()
            for keyword in SELECT_KEYWORDS:
                if keyword in keyword_text:
                    row[keyword] += 1
    except Exception as e:
        print(f"  [SELECT] 파싱 에러: {e}")

    # ── COMMENT_KEYWORDS 파싱 ────────────────────
    try:
        comments = driver.find_elements(By.CSS_SELECTOR, ".pui__vn15t2")
        print(f"  [댓글] {len(comments)}개 수집")
        for el in comments:
            comment_text = el.text
            for keyword in COMMENT_KEYWORDS:
                if keyword in comment_text:
                    row[keyword] += 1
    except Exception as e:
        print(f"  [댓글] 파싱 에러: {e}")

    finally:
        driver.quit()

    return row


def load_existing_results(output_file):
    """이미 저장된 결과 불러오기 → 완료된 식당명 집합 반환"""
    if os.path.exists(output_file):
        try:
            df_existing = pd.read_csv(output_file, encoding="utf-8-sig")
            done = set(df_existing["식당명"].tolist())
            print(f"✅ 기존 결과 파일 로드: {len(done)}개 식당 이미 완료\n")
            return df_existing, done
        except Exception as e:
            print(f"⚠️ 기존 파일 읽기 실패 ({e}) → 새로 시작\n")
    return pd.DataFrame(), set()


# ── 실행 ─────────────────────────────────────────
if __name__ == "__main__":
    rest_df = pd.read_csv("rest_link.csv", encoding="utf-8-sig")
    rest_df = rest_df.dropna(subset=["id"])
    rest_df["id"] = rest_df["id"].astype(str).str.strip()
    rest_df = rest_df[rest_df["id"] != ""]

    restaurants = rest_df.to_dict("records")
    print(f"총 {len(restaurants)}개 식당 대상\n")

    df_existing, done_names = load_existing_results(OUTPUT_FILE)
    options = get_chrome_options()
    new_rows = []

    def save_progress(reason="완료"):
        if new_rows:
            df_new = pd.DataFrame(new_rows)
            df_all = pd.concat([df_existing, df_new], ignore_index=True)
        else:
            df_all = df_existing
        df_all.to_csv(OUTPUT_FILE, encoding="utf-8-sig", index=False)
        print(f"\n💾 [{reason}] 총 {len(df_all)}개 식당 저장 → {OUTPUT_FILE}")

    try:
        for i, r in enumerate(restaurants):
            name = r["restaurant_name"]
            rest_id = r["id"]
            picture = r.get("picture", "")

            if name in done_names:
                print(f"[SKIP] {name} (이미 완료)")
                continue

            print(f"\n[START] {name} (id: {rest_id})")
            try:
                row = get_keyword_counts(name, rest_id, picture, options)
                new_rows.append(row)
                print(f"[DONE]  {name}")
            except KeyboardInterrupt:
                print(f"\n⚠️  {name} 크롤링 중 중단됨")
                raise
            except Exception as e:
                print(f"[ERROR] {name}: {e}")
                continue

            # ── 중간 저장 ────────────────────────
            df_new = pd.DataFrame(new_rows)
            df_all = pd.concat([df_existing, df_new], ignore_index=True)
            df_all.to_csv(OUTPUT_FILE, encoding="utf-8-sig", index=False)
            print(f"  → 중간 저장 완료 (누적 {len(df_all)}개)")

            # ── 식당 간 랜덤 딜레이 ──────────────
            delay = random.uniform(3.0, 7.0)
            print(f"  → {delay:.1f}초 대기 중...")
            time.sleep(delay)

            # ── 10개마다 긴 휴식 ─────────────────
            completed = sum(1 for row in new_rows)
            if completed % 10 == 0:
                long_break = random.uniform(25.0, 35.0)
                print(f"\n⏸ {completed}개 완료 — {long_break:.0f}초 휴식 (IP 차단 방지)\n")
                time.sleep(long_break)

    except KeyboardInterrupt:
        save_progress("사용자 중단 — 지금까지 저장")
    else:
        save_progress("전체 완료")

In [6]:
# ── 실행 ───────────────────────────────────────────
if __name__ == "__main__":
    # CSV 로드
    rest_df = pd.read_csv("rest_link.csv", encoding="utf-8-sig")
 
    # id가 비어있는 행 제외 (예: 역전우동0410처럼 id 없는 경우)
    rest_df = rest_df.dropna(subset=["id"])
    rest_df["id"] = rest_df["id"].astype(str).str.strip()
    rest_df = rest_df[rest_df["id"] != ""]
 
    restaurants = rest_df.to_dict("records")
    print(f"총 {len(restaurants)}개 식당 대상\n")
 
    # 기존 결과 불러오기
    df_existing, done_names = load_existing_results(OUTPUT_FILE)
 
    options = get_chrome_options()
    new_rows = []
 
    for r in restaurants:
        name = r["restaurant_name"]
        rest_id = r["id"]
        picture = r.get("picture", "")
 
        # ── 이미 완료된 식당은 건너뜀 ──
        if name in done_names:
            print(f"[SKIP] {name} (이미 완료)")
            continue
 
        print(f"\n[START] {name} (id: {rest_id})")
        try:
            row = get_keyword_counts(name, rest_id, picture, options)
            new_rows.append(row)
            print(f"[DONE]  {name}")
        except Exception as e:
            print(f"[ERROR] {name}: {e}")
            continue
 
        # ── 식당 한 개 끝날 때마다 중간 저장 ──
        df_new = pd.DataFrame(new_rows)
        df_all = pd.concat([df_existing, df_new], ignore_index=True)
        df_all.to_csv(OUTPUT_FILE, encoding="utf-8-sig", index=False)
        print(f"  → 중간 저장 완료 (누적 {len(df_all)}개)")
 
    # ── 최종 저장 ──────────────────────────────────
    if new_rows:
        df_new = pd.DataFrame(new_rows)
        df_all = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_all = df_existing
 
    df_all.to_csv(OUTPUT_FILE, encoding="utf-8-sig", index=False)
    print(f"\n✅ 완료! 총 {len(df_all)}개 식당 저장 → {OUTPUT_FILE}")
    print(df_all.set_index("식당명"))

총 153개 식당 대상


[START] 88맵도리분식포차 (id: 2074632631)
  [카테고리] 한식
  [페이지] 스크롤 에러: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: unable to send message to renderer
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x5bb593+105d3]
	chromedriver!GetHandleVerifier [0x5bb6c4+10704]
	chromedriver!(No symbol) [0x3c1ea0]
	chromedriver!(No symbol) [0x3b1379]
	chromedriver!(No symbol) [0x3b12f6]
	chromedriver!(No symbol) [0x3afe53]
	chromedriver!(No symbol) [0x3b0876]
	chromedriver!(No symbol) [0x3c6b8e]
	chromedriver!(No symbol) [0x3c7375]
	chromedriver!(No symbol) [0x3cad4a]
	chromedriver!(No symbol) [0x3cadd8]
	chromedriver!(No symbol) [0x40ad00]
	chromedriver!(No symbol) [0x40b52b]
	chromedriver!(No symbol) [0x44d052]
	chromedriver!(No symbol) [0x42d7c4]
	chro

KeyboardInterrupt: 